# Text Simplification with GPT

This notebook takes the book excerpts from `gold_standard.csv` and generates simplified versions
at different target reading ages using OpenAI's GPT models (gpt-3.5-turbo, gpt-4o-mini, gpt-5-nano, gpt-5, gpt-6-luna).

The simplified texts are scored with Flesch-Kincaid and saved back to `gold_standard.csv`
for use in `evaluate_models_on_gold_standard_data.ipynb`.

**Requirements:** An OpenAI API key set as the environment variable `OPENAI_API_KEY`.

> Note: This notebook makes API calls (8 excerpts x 7 target ages x 5 models = 280 requests).
> Approximate cost: ~£0.60 / ~$0.75 USD for the full run of 280 calls across all 5 models.
> GPT-5 output tokens dominate the cost at $10/MTok out.

In [3]:
import pandas as pd
import textstat
import re
import os
import time
from openai import OpenAI
from nltk.corpus import stopwords


## Configuration

Set your OpenAI API key as an environment variable before running:
```
export OPENAI_API_KEY=your-key-here
```

In [4]:
import getpass

# Prompts for your API key if not already set in the environment
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

client = OpenAI()

MODEL_MAP = {
    'gpt-3.5-turbo': 'gpt-3.5-turbo',
    'gpt-4o-mini':   'gpt-4o-mini',
    'gpt-5-nano':    'gpt-5-nano',
    'gpt-5':         'gpt-5',
    'gpt-6-luna':    'gpt-6-luna',
}


## Load Gold Standard

In [5]:
df = pd.read_csv('gold_standard.csv')
print(f'Loaded {len(df)} rows')
df[['title', 'author', 'target_age', 'model']].head(10)

Loaded 280 rows


,title,author,target_age,model
0,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-3.5-turbo
1,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-4o-mini
2,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-5-nano
3,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-5
4,The Tale of Peter Rabbit,Beatrix Potter,6,gpt-6-luna
5,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-3.5-turbo
6,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-4o-mini
7,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-5-nano
8,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-5
9,The Tale of Peter Rabbit,Beatrix Potter,8,gpt-6-luna


## Simplification Function

In [ ]:
def simplify_text(text, target_age, model):
    response = client.chat.completions.create(
        model=MODEL_MAP[model],
        messages=[
            {
                'role': 'system',
                'content': (
                    'Simplify the supplied text for the requested reading age. '
                    'Keep its meaning and key details. Return only the simplified text.'
                ),
            },
            {
                'role': 'user',
                'content': f'Reading age: {target_age}\n\nText:\n{text}',
            },
        ],
    )
    return response.choices[0].message.content.strip()


## Readability Enrichment

In [13]:
stop = stopwords.words('english')

def enrich(text):
    if not text or not text.strip():
        return None, None, None
    fk = textstat.flesch_kincaid_grade(text)
    age = round(fk + 5)
    tokens = [w for w in re.sub('[^A-Za-z]', ' ', text.lower()).split()
              if w not in stop and len(w) > 1]
    return fk, age, len(tokens)


## Run Simplification

Processes each row in the gold standard where simplified_text is empty.
Saves progress after every 10 rows so you can resume if interrupted.

In [ ]:
df['simplified_text'] = df['simplified_text'].astype('object')

rows_to_process = df[df['simplified_text'].isna() | (df['simplified_text'] == '')].index
print(f'{len(rows_to_process)} rows to process')

for i, idx in enumerate(rows_to_process):
    row = df.loc[idx]
    simplified = simplify_text(row['text_content'], row['target_age'], row['model'])
    fk, age, tokens = enrich(simplified)

    df.at[idx, 'simplified_text'] = simplified
    df.at[idx, 'fk_score_simplified'] = fk
    df.at[idx, 'reading_age_simplified'] = age
    df.at[idx, 'token_count_simplified'] = tokens

    if (i + 1) % 10 == 0:
        df.to_csv('gold_standard.csv', index=False)
        print(f'  Saved progress: {i + 1}/{len(rows_to_process)} rows done')

    time.sleep(0.5)

df.to_csv('gold_standard.csv', index=False)
print('Done. Saved gold_standard.csv')


## Preview Results

In [15]:
sample = df[df['simplified_text'] != ''][['title', 'target_age', 'model', 'reading_age_original', 'reading_age_simplified']].head(10)
print(sample.to_string())

                      title  target_age          model  reading_age_original  reading_age_simplified
0  The Tale of Peter Rabbit           6  gpt-3.5-turbo                    11                    11.0
1  The Tale of Peter Rabbit           6    gpt-4o-mini                    11                    10.0
2  The Tale of Peter Rabbit           6     gpt-5-nano                    11                    10.0
3  The Tale of Peter Rabbit           6          gpt-5                    11                     9.0
4  The Tale of Peter Rabbit           6     gpt-6-luna                    11                     9.0
5  The Tale of Peter Rabbit           8  gpt-3.5-turbo                    11                    12.0
6  The Tale of Peter Rabbit           8    gpt-4o-mini                    11                    10.0
7  The Tale of Peter Rabbit           8     gpt-5-nano                    11                    10.0
8  The Tale of Peter Rabbit           8          gpt-5                    11               

## Cost Estimate

In [16]:
# Cost per 1K tokens (input + output blended estimate, USD)
COST_PER_1K = {
    'gpt-3.5-turbo': 0.00200,   # $0.50 in / $1.50 out per MTok
    'gpt-4o-mini':   0.00038,   # $0.15 in / $0.60 out per MTok
    'gpt-5-nano':    0.00023,   # $0.05 in / $0.40 out per MTok
    'gpt-5':         0.00575,   # $1.25 in / $10.00 out per MTok
    'gpt-6-luna':    0.00030,   # $0.10 in / $0.50 out per MTok
}

completed = df[df['simplified_text'] != ''].copy()
completed['estimated_cost_usd'] = completed.apply(
    lambda row: ((row['token_count'] + row['token_count_simplified']) / 1000)
    * COST_PER_1K.get(row['model'], 0.0015),
    axis=1
)

cost_summary = completed.groupby('model')['estimated_cost_usd'].sum()
print('Estimated cost by model (USD):')
print(cost_summary)

Estimated cost by model (USD):
model
gpt-3.5-turbo    0.011884
gpt-4o-mini      0.002802
gpt-5            0.045109
gpt-5-nano       0.001812
gpt-6-luna       0.002405
Name: estimated_cost_usd, dtype: float64
